In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
if os.environ.get("BIRDDOG_USE_LOCAL_NOCODB"):
    print("using local nocodb")
else:
    print("using aws nocodb")

using aws nocodb


In [3]:
from birddog.runtime import Runtime
from birddog.database import Database
from birddog.database_updater import (
    DatabaseUpdater,
    _fetch_mediawiki_file_metadata,
    _source_from_url,
    form_document_record,
    )

2026-06-01 17:29:11,887 [INFO] Using local folder /Users/jbrandt/code/birddog/.cache for storage.
2026-06-01 17:29:12,021 [INFO] Translation is enabled. Using GCP translator
2026-06-01 17:29:12,022 [INFO] Using Google Cloud translation API
2026-06-01 17:29:12,022 [INFO] GoogleCloudTranslator using REST API
2026-06-01 17:29:12,366 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com


In [11]:
db = Database()

2026-06-01 17:30:41,627 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   21.00     0.04    39.00       0.00           24
  uk.wikisource.org:api                  4.00     0.03     4.00       0.00            4


In [5]:
runtime = Runtime()

2026-06-01 17:29:28,347 [INFO] PageUpdateManager.init(): detect_environment==local
2026-06-01 17:29:29,011 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   20.00     4.88    39.00       0.00           24
2026-06-01 17:29:29,520 [INFO] WikiDocTracker: base=https://commons.wikimedia.org, namespace=File (id=6)
2026-06-01 17:29:29,932 [INFO] WikiDocTracker: base=https://uk.wikisource.org, namespace=Файл (id=6)
2026-06-01 17:29:29,934 [INFO] KillSwitch: loading thresholds from resources/kill_thresholds.json
2026-06-01 17:29:29,936 [INFO] Runtime: truncating log history before 2026-05-17 23:29:29.936252+00:00


In [6]:
#doc_url="https://upload.wikimedia.org/wikipedia/commons/archive/d/d7/20230312214857!R5069-32-0002.pdf"

In [7]:
#_source_from_url(doc_url)

In [8]:
#form_document_record(doc_url)

In [ ]:
t= form_document_record(doc_url)['title']
mr=_fetch_mediawiki_file_metadata([t],source="commons")
mr.get(t)

In [21]:
def do_pass(limit=5, source="commons"):
    rec, _ = db.scan("Documents", view_name="BD:Missing Metadata", fields="url", limit=limit)
    doc_recs = [form_document_record(r["url"]) for r in rec]
    #return doc_recs
    subset = [rec for rec in doc_recs if _source_from_url(rec["url"]) == source]
    subset_titles = [rec["title"] for rec in subset]
    #subset_titles.extend(["|", "[", "]", ""])
    #subset_titles.append("MISSING_TITLE")
    #return subset_titles
    metadata_recs = _fetch_mediawiki_file_metadata(subset_titles, source)
    return metadata_recs

In [14]:
def do_pass_2(limit=10):
    rec, _ = db.scan("Documents", view_name="BD:Missing Metadata", fields="url", limit=limit)
    urls = [r["url"] for r in rec]
    return urls
    if urls:
        print(f"updating document metadata len=={len(urls)}")
        runtime.update_documents_to_database(urls)

In [16]:
urls = do_pass_2(limit=100)

2026-06-01 17:37:58,733 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   24.00     0.02    39.00       0.00           24
  uk.wikisource.org:api                  4.00     0.00     4.00       0.00            4


In [ ]:
runtime.run_database_housekeeping()

In [ ]:
runtime._database_updater.update_doc_records([doc_url])

In [ ]:
def update_wiki_flag(db):
    limit = 1000
    while True:
        recs, _ = db.scan("Documents", limit=limit, view_name="Wiki Flag Not Set", fields="url")
        recs = [r for r in recs if "wikimedia" in r['url'] or "wikisource" in r['url']]
        for r in recs:
            r['wiki'] = True
        if not recs:
            break
        print(len(recs))
        db.write("Documents", recs)            

In [ ]:
update_wiki_flag(db)

In [24]:
runtime._database_updater.update_doc_records(urls[:1])

2026-06-01 17:54:13,262 [INFO] Updater.update_doc_records 8328257152: #docs=1, metadata update=True
2026-06-01 17:54:13,559 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  commons.wikimedia.org:api              4.50     0.00     4.00       0.00            4
  nocodb.internal:api                   27.00     0.00    39.00       0.00           24
  uk.wikisource.org:api                  4.00     0.00     4.00       0.00            4
2026-06-01 17:54:14,162 [INFO] Updater.update_doc_records 8328257152: finished


True

In [22]:
do_pass(limit=1)

{'File:ДАВіО_208-1-671_Вироки_сільських_сходів_Гавришівської_волості_про_затвердження_призовних_списків_(1893).pdf': {'title': 'File:ДАВіО 208-1-671 Вироки сільських сходів Гавришівської волості про затвердження призовних списків (1893).pdf',
  'timestamp': '2026-06-01 23:36:06+00:00',
  'byte_size': 81264350,
  'mimetype': 'application/pdf',
  'mediatype': 'OFFICE',
  'width': 1675,
  'height': 2285,
  'page_count': 41,
  'sha1_hash': '470fd7f575a8573cf18bf0311a7d1e7e515d5359',
  'description_url': 'https://commons.wikimedia.org/wiki/File:%D0%94%D0%90%D0%92%D1%96%D0%9E_208-1-671_%D0%92%D0%B8%D1%80%D0%BE%D0%BA%D0%B8_%D1%81%D1%96%D0%BB%D1%8C%D1%81%D1%8C%D0%BA%D0%B8%D1%85_%D1%81%D1%85%D0%BE%D0%B4%D1%96%D0%B2_%D0%93%D0%B0%D0%B2%D1%80%D0%B8%D1%88%D1%96%D0%B2%D1%81%D1%8C%D0%BA%D0%BE%D1%97_%D0%B2%D0%BE%D0%BB%D0%BE%D1%81%D1%82%D1%96_%D0%BF%D1%80%D0%BE_%D0%B7%D0%B0%D1%82%D0%B2%D0%B5%D1%80%D0%B4%D0%B6%D0%B5%D0%BD%D0%BD%D1%8F_%D0%BF%D1%80%D0%B8%D0%B7%D0%BE%D0%B2%D0%BD%D0%B8%D1%85_%D1%81%D0%BF%D0

In [23]:
urls[0]

'https://commons.wikimedia.org/wiki/File:ДАВіО_208-1-671_Вироки_сільських_сходів_Гавришівської_волості_про_затвердження_призовних_списків_(1893).pdf'